In [1]:
import os
from dotenv import load_dotenv
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from dataclasses import dataclass

from google.cloud import storage

import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType, StructType, StructField, IntegerType, StringType, DateType

from utils.LocationFunctions import load_locations_df, get_locations_from_bq, get_missing_locations, get_batch_geocode, update_locations_bq
from utils.Common import gcs_file_read, gcs_upload_parquet, partial_parse_raw_data, parse_involved_data
from utils.VehicleFunctions import load_vehicles_df, add_vehicle_suggestions

gcs_connector_path = '../../config/gcs-connector-hadoop3-latest.jar'
bigquery_connector_path = '../../config/spark-bigquery-with-dependencies_2.12-0.35.0.jar'

load_dotenv()

ScrapedRawSchema = StructType([
    StructField('content', StringType(), True),
    StructField('tweetlinkid', StringType(), True),
    StructField('created_at', DateType(), True),
])

KaggleRawSchema = StructType([
    StructField('Date', DateType(), True),
    StructField('Time', StringType(), True),
    StructField('City', StringType(), True),
    StructField('Location', StringType(), True),
    StructField('Latitude', DoubleType(), True),
    StructField('Longitude', DoubleType(), True),
    StructField('High_Accuracy', DoubleType(), True),
    StructField('Direction', StringType(), True),
    StructField('Type', StringType(), True),
    StructField('Lanes_Blocked', IntegerType(), True),
    StructField('Involved', StringType(), True),
    StructField('Tweet', StringType(), True),
    StructField('Source', StringType(), True),
])

@dataclass
class TransformContext:
    spark: any
    gcs_client: any
    project_id: str
    dataset: str
    locations_table_id: str
    vehicle_table_id: str
    staging_locations_table_id: str
    bucket_name: str
    raw_folder: str
    clean_folder: str
    vehicles_folder: str
    scrape_folder: str
    df_locations: any
    df_vehicles: any

def initialize(app_name, data_source='scraped'):
    """Initialize Spark, GCS client, and location references for the transform pipeline.

    Args:
        app_name: Name for the Spark application.
        data_source: Data source type ('scraped' or 'kaggle').

    Returns:
        TransformContext: spark, gcs_client, project_id, dataset, locations_table_id,
               staging_locations_table_id, bucket_name, raw_folder,
               clean_folder, scrape_folder, df_locations, df_vehicles
    """
    spark = SparkSession.builder \
            .master("local[*]") \
            .appName(app_name) \
            .config("spark.jars", f"{gcs_connector_path},{bigquery_connector_path}") \
            .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem") \
            .config("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
            .getOrCreate()
    
    gcs_client = storage.Client()

    project_id = os.getenv("PROJECT_ID")
    dataset = os.getenv("DATASET")
    locations_table_id = f"{project_id}:{dataset}.dim_locations"
    vehicle_table_id = f"{project_id}:{dataset}.dim_vehicle"
    staging_locations_table_id = f"{project_id}:{dataset}.staging_locations"
    bucket_name = os.getenv('BUCKET_NAME')
    raw_folder = os.getenv('RAW_FOLDER_NAME')
    clean_folder = os.getenv('CLEANED_FOLDER_NAME')
    vehicles_folder = os.getenv('VEHICLES_FOLDER_NAME')
    scrape_folder = f"{raw_folder}/scrape"

    df_locations = load_locations_df(spark, locations_table_id)
    df_vehicles = load_vehicles_df(spark, vehicle_table_id)

    return TransformContext(
        spark=spark,
        gcs_client=gcs_client,
        project_id=project_id,
        dataset=dataset,
        locations_table_id=locations_table_id,
        vehicle_table_id=vehicle_table_id,
        staging_locations_table_id=staging_locations_table_id,
        bucket_name=bucket_name,
        raw_folder=raw_folder,
        clean_folder=clean_folder,
        vehicles_folder=vehicles_folder,
        scrape_folder=scrape_folder,
        df_locations=df_locations,
        df_vehicles=df_vehicles,
    )

    
def generate_filenames(scrape_folder, start_date, end_date):
    """
    Generates list of GCS filenames between start_date and end_date (inclusive)
    
    Args:
        scrape_folder: GCS folder path for scrape data
        start_date: Start date string in "YYYY-MM-DD" format
        end_date: End date string in "YYYY-MM-DD" format
        
    Returns:
        List of GCS file paths
    """
    PHT = ZoneInfo("Asia/Manila")

    start = datetime.strptime(start_date, "%Y-%m-%d").replace(tzinfo=PHT)
    end = datetime.strptime(end_date, "%Y-%m-%d").replace(tzinfo=PHT)

    current = start
    filenames = []

    while current <= end:
        filenames.append(build_raw_filename(scrape_folder, current))
        current += timedelta(days=1)

    return filenames


def get_current_raw_filename(scrape_folder):
    """
    Get the GCS filename for yesterday's scrape data
    
    Args:
        scrape_folder: GCS folder path for scrape data
        
    Returns:
        GCS file path for yesterday's data
    """
    PHT = ZoneInfo("Asia/Manila")
    now = datetime.now(PHT)
    yesterday = now - timedelta(days=1)

    return build_raw_filename(scrape_folder, yesterday)


def build_raw_filename(scrape_folder, dt):
    """Builds a raw scrape filename from a datetime-like object.

    Args:
        scrape_folder: base folder path
        dt: datetime or date object

    Returns:
        filename string
    """
    year = dt.strftime("%Y")
    month = dt.strftime("%m")
    day = dt.strftime("%d")

    return f"{scrape_folder}/{year}/{month}/scrape_data_{year}{month}{day}.csv"

In [2]:
ctx = initialize('Transform Stage (Daily)', 'scraped')

raw_filename = get_current_raw_filename(ctx.scrape_folder)

26/06/24 11:49:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [3]:
def load_raw_if_exists(spark, gcs_client, bucket_name, raw_filename, schema):
    """Helper to check for GCS blob existence and read it.

    Returns DataFrame if exists, otherwise None.
    """
    bucket = gcs_client.bucket(bucket_name)
    blob = storage.Blob(bucket=bucket, name=raw_filename)
    if not blob.exists():
        print(f"Skipping (not found): {raw_filename}")
        return None

    print(f"Processing: {raw_filename}")
    return gcs_file_read(spark, bucket_name, raw_filename, schema)

In [4]:
df_raw = load_raw_if_exists(ctx.spark, ctx.gcs_client, ctx.bucket_name, raw_filename, ScrapedRawSchema)

Processing: raw/scrape/2026/06/scrape_data_20260623.csv


In [5]:
df_partial_parsed = partial_parse_raw_data(df_raw)

# handle missing timestamps
null_timestamp_count = df_partial_parsed.filter(F.col("event_timestamp").isNull()).count()
if null_timestamp_count > 0:
    window = Window.rowsBetween(Window.unboundedPreceding, 0)
    df_partial_parsed = df_partial_parsed.withColumn("time", F.last("time", ignorenulls=True).over(window))
    df_partial_parsed = df_partial_parsed.withColumn(
        "event_timestamp", 
        F.to_timestamp(F.concat_ws(' ', F.col("date"), F.col("time")), "yyyy-MM-dd HH:mm")
    )

# enrich from BQ
# df_full_parsed = get_locations_from_bq(ctx.df_locations, df_partial_parsed)

In [ ]:
df_partial_parsed.show(5, truncate=False)

In [6]:
df_involved = parse_involved_data(df_partial_parsed)

In [7]:
df_involved.show(5, truncate=False)

+----------------------------------------------------------------+------------+-------------+
|event_id                                                        |vehicle_type|vehicle_count|
+----------------------------------------------------------------+------------+-------------+
|E24022F3921C9B3019706EA7916951B2A7839DE8E1427481B4D36186749206B0|AUV         |1            |
|E24022F3921C9B3019706EA7916951B2A7839DE8E1427481B4D36186749206B0|L300        |1            |
|04B91A2F25789D3699BF4FD28E8F58DAC773057E5C184E0B6D8BF2EB63B0AE9C|CAR         |1            |
|04B91A2F25789D3699BF4FD28E8F58DAC773057E5C184E0B6D8BF2EB63B0AE9C|TAXI        |1            |
|C2821B0CEED67DDDA65CD90EAD0710301DF1E0603D572943593DBDE67EC0809F|CAR         |1            |
+----------------------------------------------------------------+------------+-------------+
only showing top 5 rows



In [27]:
verified_lookup = (
        ctx.df_vehicles
        .select(
            F.upper(F.trim("vehicle_type")).alias("vehicle_type"),
            "vehicle_group",
            "is_verified"
        )
    )
verified_lookup.show(5, truncate=False)

+----------------+-------------+-----------+
|vehicle_type    |vehicle_group|is_verified|
+----------------+-------------+-----------+
|DUMP TRUCK      |COMMERCIAL   |true       |
|10-WHEELER TRUCK|COMMERCIAL   |true       |
|CLOSED VAN      |COMMERCIAL   |true       |
|TRUCK           |COMMERCIAL   |true       |
|L300            |COMMERCIAL   |true       |
+----------------+-------------+-----------+
only showing top 5 rows



In [28]:
staging_vehicle_types = (
        df_involved
        .select(F.upper(F.trim("vehicle_type")).alias("vehicle_type"))
        .distinct()
    )
staging_vehicle_types.show(5, truncate=False)

+------------+
|vehicle_type|
+------------+
|UV EXPRESS  |
|PEDESTRIAN  |
|CAR         |
|BUS         |
|PUJ         |
+------------+
only showing top 5 rows



In [29]:
verified_types = (
        verified_lookup
        .filter(F.col("is_verified") == True)
        .select(
            F.col("vehicle_type").alias("candidate_vehicle_type"),
            "vehicle_group"
        )
    )
verified_types.show(5, truncate=False)

+----------------------+-------------+
|candidate_vehicle_type|vehicle_group|
+----------------------+-------------+
|DUMP TRUCK            |COMMERCIAL   |
|10-WHEELER TRUCK      |COMMERCIAL   |
|CLOSED VAN            |COMMERCIAL   |
|TRUCK                 |COMMERCIAL   |
|L300                  |COMMERCIAL   |
+----------------------+-------------+
only showing top 5 rows



In [30]:
unknown = (
        staging_vehicle_types
        .join(
            verified_types,
            staging_vehicle_types.vehicle_type == verified_types.candidate_vehicle_type,
            "left_anti"
        )
    )
unknown.show(5, truncate=False)

+-------------+
|vehicle_type |
+-------------+
|TRICYCLE     |
|WING VAN     |
|TRAILER TRUCK|
+-------------+



In [31]:
candidates = (
        unknown
        .crossJoin(F.broadcast(verified_types))
        .withColumn(
            "distance",
            F.levenshtein("vehicle_type", "candidate_vehicle_type")
        )
        .withColumn(
            "max_len",
            F.greatest(
                F.length("vehicle_type"),
                F.length("candidate_vehicle_type")
            )
        )
        .withColumn(
            "similarity",
            1 - (F.col("distance") / F.col("max_len"))
        )
        .withColumn(
            "contains_bonus",
            F.when(
                F.col("vehicle_type").contains(F.col("candidate_vehicle_type"))
                | F.col("candidate_vehicle_type").contains(F.col("vehicle_type")),
                F.lit(0.15)
            ).otherwise(F.lit(0.0))
        )
        .withColumn(
            "match_score",
            F.col("similarity") + F.col("contains_bonus")
        )
    )
candidates.show(5, truncate=False)

+------------+----------------------+-------------+--------+-------+----------+--------------+-----------+
|vehicle_type|candidate_vehicle_type|vehicle_group|distance|max_len|similarity|contains_bonus|match_score|
+------------+----------------------+-------------+--------+-------+----------+--------------+-----------+
|TRICYCLE    |DUMP TRUCK            |COMMERCIAL   |10      |10     |0.0       |0.0           |0.0        |
|TRICYCLE    |10-WHEELER TRUCK      |COMMERCIAL   |14      |16     |0.125     |0.0           |0.125      |
|TRICYCLE    |CLOSED VAN            |COMMERCIAL   |10      |10     |0.0       |0.0           |0.0        |
|TRICYCLE    |TRUCK                 |COMMERCIAL   |5       |8      |0.375     |0.0           |0.375      |
|TRICYCLE    |L300                  |COMMERCIAL   |8       |8      |0.0       |0.0           |0.0        |
+------------+----------------------+-------------+--------+-------+----------+--------------+-----------+
only showing top 5 rows



In [33]:
w = Window.partitionBy("vehicle_type").orderBy(F.desc("match_score"))

best_suggestion = (
    candidates
    .withColumn("rn", F.row_number().over(w))
    .filter(F.col("rn") == 1)
    .select(
        "vehicle_type",
        F.col("candidate_vehicle_type").alias("suggested_from")
    )
)

In [34]:
verified_rows = (
        df_involved
        .withColumn("vehicle_type", F.upper(F.trim("vehicle_type")))
        .join(
            verified_lookup.filter(F.col("is_verified") == True),
            "vehicle_type",
            "inner"
        )
        .withColumn("suggested_from", F.lit(None).cast("string"))
        .select(
            "event_id",
            "vehicle_type",
            "vehicle_count",
            "vehicle_group",
            "suggested_from",
            "is_verified"
        )
    )
verified_rows.show(5, truncate=False)

+----------------------------------------------------------------+------------+-------------+-------------+--------------+-----------+
|event_id                                                        |vehicle_type|vehicle_count|vehicle_group|suggested_from|is_verified|
+----------------------------------------------------------------+------------+-------------+-------------+--------------+-----------+
|3C473BECB5C072F7920F4A4F98C9C915F69F044C639DA5110BCEF74436A8D01F|DUMP TRUCK  |1            |COMMERCIAL   |null          |true       |
|DB00430D435E391679504FFF4863BAC341F78164DE36639BEC20E9BBC0870FEE|DUMP TRUCK  |1            |COMMERCIAL   |null          |true       |
|C06F34018F9E7151AE5897F909EDA3F6E29DC3628EB9054BD461075DEA316E0E|CLOSED VAN  |1            |COMMERCIAL   |null          |true       |
|1F2107BD1AFA206F9D43351A4598F7387E22A73B285E176173BF6EDDFB49DD44|TRUCK       |1            |COMMERCIAL   |null          |true       |
|DE28D0D2EC053DFE6423C4A70CC860D4428E1CC36B7664DDE58EE1

In [35]:
unverified_rows = (
        df_involved
        .withColumn("vehicle_type", F.upper(F.trim("vehicle_type")))
        .join(
            verified_lookup.filter(F.col("is_verified") == True)
                          .select("vehicle_type"),
            "vehicle_type",
            "left_anti"
        )
        .join(best_suggestion, "vehicle_type", "left")
        .join(
            verified_types.select(
                F.col("candidate_vehicle_type").alias("suggested_from"),
                "vehicle_group"
            ),
            "suggested_from",
            "left"
        )
        .withColumn("vehicle_group", F.coalesce("vehicle_group", F.lit("UNKNOWN")))
        .withColumn("is_verified", F.lit(False))
        .select(
            "event_id",
            "vehicle_type",
            "vehicle_count",
            "vehicle_group",
            "suggested_from",
            "is_verified"
        )
    )

unverified_rows.show(5, truncate=False)

+----------------------------------------------------------------+-------------+-------------+-------------+--------------+-----------+
|event_id                                                        |vehicle_type |vehicle_count|vehicle_group|suggested_from|is_verified|
+----------------------------------------------------------------+-------------+-------------+-------------+--------------+-----------+
|C07B8C2293CEC816D931539634CB2F6EED3614193E1938D44B9CD980FD49D4D4|WING VAN     |1            |PRIVATE      |VAN           |false      |
|19E20160F1E843EB7416487A72600C29C3BF241E24507978FBCA060175D1D92E|WING VAN     |1            |PRIVATE      |VAN           |false      |
|6118D3CD0A14B99AAA2FB3F4DAD9C19920F4F20F288DC5494697F32B31770217|TRAILER TRUCK|1            |COMMERCIAL   |TRAILER       |false      |
|5353BE9CC0BA2CEAAFA3DF1DB1CFF9AB3D6F5C5EE8EE30005E7CA498FAD1995F|TRAILER TRUCK|1            |COMMERCIAL   |TRAILER       |false      |
|ADF6F6994B7EAD593C200F76078EB6264BD14ACEAD22D6B

In [36]:
df_involved_enriched = verified_rows.unionByName(unverified_rows)

+----------------------------------------------------------------+------------+-------------+-------------+--------------+-----------+
|event_id                                                        |vehicle_type|vehicle_count|vehicle_group|suggested_from|is_verified|
+----------------------------------------------------------------+------------+-------------+-------------+--------------+-----------+
|3C473BECB5C072F7920F4A4F98C9C915F69F044C639DA5110BCEF74436A8D01F|DUMP TRUCK  |1            |COMMERCIAL   |null          |true       |
|DB00430D435E391679504FFF4863BAC341F78164DE36639BEC20E9BBC0870FEE|DUMP TRUCK  |1            |COMMERCIAL   |null          |true       |
|C06F34018F9E7151AE5897F909EDA3F6E29DC3628EB9054BD461075DEA316E0E|CLOSED VAN  |1            |COMMERCIAL   |null          |true       |
|1F2107BD1AFA206F9D43351A4598F7387E22A73B285E176173BF6EDDFB49DD44|TRUCK       |1            |COMMERCIAL   |null          |true       |
|DE28D0D2EC053DFE6423C4A70CC860D4428E1CC36B7664DDE58EE1

In [37]:
df_involved_enriched.show(truncate=False)

+----------------------------------------------------------------+------------+-------------+-------------+--------------+-----------+
|event_id                                                        |vehicle_type|vehicle_count|vehicle_group|suggested_from|is_verified|
+----------------------------------------------------------------+------------+-------------+-------------+--------------+-----------+
|3C473BECB5C072F7920F4A4F98C9C915F69F044C639DA5110BCEF74436A8D01F|DUMP TRUCK  |1            |COMMERCIAL   |null          |true       |
|DB00430D435E391679504FFF4863BAC341F78164DE36639BEC20E9BBC0870FEE|DUMP TRUCK  |1            |COMMERCIAL   |null          |true       |
|C06F34018F9E7151AE5897F909EDA3F6E29DC3628EB9054BD461075DEA316E0E|CLOSED VAN  |1            |COMMERCIAL   |null          |true       |
|1F2107BD1AFA206F9D43351A4598F7387E22A73B285E176173BF6EDDFB49DD44|TRUCK       |1            |COMMERCIAL   |null          |true       |
|DE28D0D2EC053DFE6423C4A70CC860D4428E1CC36B7664DDE58EE1

In [39]:
df_involved_enriched.write \
    .mode("overwrite") \
    .option("header", "true") \
    .option("delimiter", ",") \
    .csv("test_outputs")